# Mejora de imagen y ecualización básica

En este cuaderno vas a trabajar con operaciones simples para mejorar una imagen: cambios de brillo, cambios de contraste y una primera ecualización. La idea es entender qué se gana y qué se arriesga con cada decisión.


## Objetivo

Analizar cómo pequeñas transformaciones de intensidad cambian la lectura visual de una imagen y reconocer cuándo una mejora realmente ayuda y cuándo empieza a deformar la escena.

## Resultados de aprendizaje

Al finalizar este cuaderno, vas a poder:

- ajustar brillo y contraste con una transformación simple;
- aplicar ecualización global sobre una imagen en escala de grises;
- comparar una mejora en RGB con una mejora aplicada sobre el canal de brillo en HSV;
- describir limitaciones básicas de cada estrategia.

## Relación con la secuencia

Este cuaderno continúa la introducción a espacios de color. Después de entender cómo representar una imagen, ahora el foco está en intervenir esa representación para volver algunos detalles más visibles.


## Módulos que vamos a usar

- `cv2`: para convertir espacios de color y aplicar transformaciones de intensidad.
- `numpy`: para limitar valores y trabajar con arreglos.
- `matplotlib.pyplot`: para comparar visualmente resultados.
- `pathlib.Path`: para acceder a la imagen de trabajo.


In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

ruta_imagen = Path("Imagenes") / "valeria.png"
imagen_bgr = cv2.imread(str(ruta_imagen), cv2.IMREAD_COLOR)
if imagen_bgr is None:
    raise FileNotFoundError(f"No se pudo leer la imagen: {ruta_imagen}")

imagen_rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
print(f"Dimensiones: {imagen_rgb.shape}")


## 1. Imagen original

Primero conviene mirar la imagen tal como está. Recién después tiene sentido preguntar si necesita más brillo, más contraste o una redistribución distinta de intensidades.


In [ ]:
plt.figure(figsize=(8, 6), constrained_layout=True)
plt.imshow(imagen_rgb)
plt.title("Imagen original", fontweight="bold", loc="left")
plt.axis("off")
plt.show()


## 2. Ajustar brillo y contraste con una transformación simple

Vamos a usar una operación lineal muy habitual:

`nuevo_valor = contraste * valor_original + brillo`

Esto no “entiende” la escena. Solo modifica intensidades. Por eso conviene observar con cuidado qué mejora y qué empeora.


In [ ]:
def ajustar_brillo_contraste(imagen, brillo=0, contraste=1.0):
    '''Ajusta brillo y contraste con una transformación lineal.'''
    return cv2.convertScaleAbs(imagen, alpha=contraste, beta=brillo)

imagen_mas_brillo = ajustar_brillo_contraste(imagen_rgb, brillo=35, contraste=1.0)
imagen_menos_brillo = ajustar_brillo_contraste(imagen_rgb, brillo=-35, contraste=1.0)
imagen_mas_contraste = ajustar_brillo_contraste(imagen_rgb, brillo=0, contraste=1.35)
imagen_menos_contraste = ajustar_brillo_contraste(imagen_rgb, brillo=0, contraste=0.75)


In [ ]:
fig, ejes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)

ejes[0, 0].imshow(imagen_rgb)
ejes[0, 0].set_title("Original", fontweight="bold", loc="left")
ejes[0, 0].axis("off")

ejes[0, 1].imshow(imagen_mas_brillo)
ejes[0, 1].set_title("Más brillo", fontweight="bold", loc="left")
ejes[0, 1].axis("off")

ejes[0, 2].imshow(imagen_menos_brillo)
ejes[0, 2].set_title("Menos brillo", fontweight="bold", loc="left")
ejes[0, 2].axis("off")

ejes[1, 0].imshow(imagen_mas_contraste)
ejes[1, 0].set_title("Más contraste", fontweight="bold", loc="left")
ejes[1, 0].axis("off")

ejes[1, 1].imshow(imagen_menos_contraste)
ejes[1, 1].set_title("Menos contraste", fontweight="bold", loc="left")
ejes[1, 1].axis("off")

ejes[1, 2].axis("off")

plt.show()


Fijate que aumentar brillo y aumentar contraste no son lo mismo. En un caso movés toda la imagen hacia tonos más claros u oscuros. En el otro, separás más o menos las diferencias entre regiones.


## 3. Ecualización global en escala de grises

La ecualización busca redistribuir intensidades para ocupar mejor el rango disponible. Para una primera aproximación, conviene hacerlo sobre una imagen en escala de grises.


In [ ]:
imagen_gris = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2GRAY)
imagen_gris_ecualizada = cv2.equalizeHist(imagen_gris)


In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
ejes[0].imshow(imagen_gris, cmap="gray")
ejes[0].set_title("Grises original", fontweight="bold", loc="left")
ejes[0].axis("off")

ejes[1].imshow(imagen_gris_ecualizada, cmap="gray")
ejes[1].set_title("Grises ecualizada", fontweight="bold", loc="left")
ejes[1].axis("off")
plt.show()


In [ ]:
plt.figure(figsize=(9, 4), constrained_layout=True)
plt.hist(imagen_gris.ravel(), bins=256, alpha=0.5, label="Original", color="gray")
plt.hist(imagen_gris_ecualizada.ravel(), bins=256, alpha=0.5, label="Ecualizada", color="tab:blue")
plt.title("Histogramas en escala de grises", fontweight="bold", loc="left")
plt.xlabel("Intensidad")
plt.ylabel("Cantidad de píxeles")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


La ecualización global puede aumentar el contraste percibido, pero también puede exagerar diferencias que no siempre son útiles. Por eso no conviene pensarla como una mejora automática garantizada.


## 4. Mejorar solo el brillo en HSV

En vez de tocar directamente los tres canales de color, ahora vamos a modificar solo el canal `V` de HSV. Esa estrategia suele preservar mejor la apariencia cromática general.


In [ ]:
imagen_hsv = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2HSV)
canal_h, canal_s, canal_v = cv2.split(imagen_hsv)

canal_v_ecualizado = cv2.equalizeHist(canal_v)
imagen_hsv_mejorada = cv2.merge([canal_h, canal_s, canal_v_ecualizado])
imagen_mejorada_rgb = cv2.cvtColor(imagen_hsv_mejorada, cv2.COLOR_HSV2RGB)


In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
ejes[0].imshow(imagen_rgb)
ejes[0].set_title("Original", fontweight="bold", loc="left")
ejes[0].axis("off")

ejes[1].imshow(imagen_mas_contraste)
ejes[1].set_title("Ajuste lineal", fontweight="bold", loc="left")
ejes[1].axis("off")

ejes[2].imshow(imagen_mejorada_rgb)
ejes[2].set_title("Ecualización sobre V", fontweight="bold", loc="left")
ejes[2].axis("off")
plt.show()


Acá el punto no es elegir un “ganador” universal, sino comparar criterios. Si la mejora vuelve visible información útil sin deformar demasiado el color, puede ser razonable. Si introduce efectos poco naturales, conviene revisar la estrategia.


## Limitaciones y próximos pasos

En este cuaderno viste estrategias básicas. Más adelante vas a comparar métodos más específicos, como distintas formas de ecualización y variantes adaptativas. Esa comparación tiene sentido recién después de entender estas transformaciones elementales.


## Actividad breve

Probá dos combinaciones distintas de `brillo` y `contraste` sobre esta misma imagen. Después respondé:

1. ¿qué detalle mejora con tu ajuste?
2. ¿qué parte de la imagen se degrada o se vuelve menos natural?
3. ¿preferís el ajuste lineal o la ecualización sobre `V` para este caso? Justificá tu respuesta.


## Cierre

Mejorar una imagen no significa simplemente volverla más brillante o más contrastada. Cada intervención destaca algunas diferencias y aplana otras. Por eso, antes de aplicar una técnica, conviene preguntarse qué información querés hacer visible y para qué tarea la necesitás.
